# Claude Agent SDK with Persistent Memory

Build a Claude agent that remembers across sessions using Hindsight memory tools and automatic hooks.

## Features
- In-process MCP server with retain, recall, and reflect tools
- Automatic memory hooks that inject context before each prompt
- Auto-retain agent results for future sessions
- Knowledge that compounds over repeated runs

## Prerequisites
- **Claude Code CLI** installed and authenticated:
  ```bash
  npm install -g @anthropic-ai/claude-code
  claude auth login  # or set ANTHROPIC_API_KEY below
  ```
- An LLM API key for Hindsight (OpenAI, Gemini, etc.)
- Hindsight running locally via Docker (see setup below)
- Alternatively, a [Hindsight Cloud](https://ui.hindsight.vectorize.io/signup) account (no Docker needed)

> **Note:** The Claude Agent SDK runs the Claude Code CLI as a subprocess. You need either `claude auth login` or `ANTHROPIC_API_KEY` set in your environment. Just having an API key without the CLI installed will not work.

## Start Hindsight Locally

Before running this notebook, start Hindsight in a terminal:

```bash
export LLM_API_KEY="your-llm-api-key"

docker run --rm -it --pull always -p 8888:8888 -p 9999:9999 \
  -e HINDSIGHT_API_LLM_API_KEY=$LLM_API_KEY \
  -e HINDSIGHT_API_LLM_MODEL=gpt-4o-mini \
  -v $HOME/.hindsight-docker:/home/hindsight/.pg0 \
  ghcr.io/vectorize-io/hindsight:latest
```

## 1. Install Dependencies

In [ ]:
!pip install -q hindsight-claude-agent-sdk nest-asyncio

## 2. Configure Environment

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import getpass

# Set your Anthropic API key (needed for Claude Agent SDK)
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

# Hindsight connection (defaults to local self-hosted instance)
HINDSIGHT_API_URL = os.getenv("HINDSIGHT_API_URL", "http://localhost:8888")
HINDSIGHT_API_KEY = os.getenv("HINDSIGHT_API_KEY", None)
BANK_ID = "claude-agent-demo"

print(f"Hindsight API: {HINDSIGHT_API_URL}")
print(f"Bank ID: {BANK_ID}")

## 3. Create a Memory Bank

In [ ]:
from hindsight_client import Hindsight

hindsight = Hindsight(base_url=HINDSIGHT_API_URL, api_key=HINDSIGHT_API_KEY)

# Create a dedicated bank for this demo (safe to re-run)
try:
    hindsight.create_bank(
        bank_id=BANK_ID,
        name="Claude Agent Demo",
        mission="Remember user preferences, decisions, and project context for a software development assistant.",
    )
    print(f"Bank '{BANK_ID}' created.")
except Exception:
    print(f"Bank '{BANK_ID}' already exists, continuing.")

## 4. Set Up Memory Tools

Create an in-process MCP server with retain, recall, and reflect tools:

In [ ]:
from hindsight_claude_agent_sdk import create_hindsight_server

server = create_hindsight_server(
    bank_id=BANK_ID,
    hindsight_api_url=HINDSIGHT_API_URL,
    api_key=HINDSIGHT_API_KEY,
    tags=["source:claude-agent-sdk-demo"],
)

print("Hindsight MCP server created with tools: hindsight_retain, hindsight_recall, hindsight_reflect")

## 5. Run Agent with Explicit Memory Tools

The agent decides when to store and retrieve memories:

In [ ]:
import subprocess
import sys
import tempfile
import time
import os

def run_agent(prompt: str, system: str = None, use_hooks: bool = False, retries: int = 3):
    """Run a Claude agent in a subprocess (Claude Agent SDK requires asyncio.run)."""
    lines = []
    lines.append("import asyncio")
    lines.append("from claude_agent_sdk import query, ClaudeAgentOptions")
    lines.append("from hindsight_claude_agent_sdk import create_hindsight_server")
    lines.append("")
    lines.append("server = create_hindsight_server(")
    lines.append(f"    bank_id={BANK_ID!r},")
    lines.append(f"    hindsight_api_url={HINDSIGHT_API_URL!r},")
    if HINDSIGHT_API_KEY:
        lines.append(f"    api_key={HINDSIGHT_API_KEY!r},")
    lines.append('    tags=["source:claude-agent-sdk-demo"],')
    lines.append(")")
    lines.append("")

    if use_hooks:
        lines.append("from hindsight_claude_agent_sdk import create_memory_hooks, MemoryHookConfig")
        lines.append("hooks = create_memory_hooks(")
        lines.append(f"    bank_id={BANK_ID!r},")
        lines.append(f"    hindsight_api_url={HINDSIGHT_API_URL!r},")
        if HINDSIGHT_API_KEY:
            lines.append(f"    api_key={HINDSIGHT_API_KEY!r},")
        lines.append("    hook_config=MemoryHookConfig(")
        lines.append("        auto_recall=True,")
        lines.append("        auto_retain=True,")
        lines.append("        recall_max_results=5,")
        lines.append("    ),")
        lines.append(")")
        lines.append("")

    lines.append("async def main():")
    lines.append("    options = ClaudeAgentOptions(")
    lines.append('        mcp_servers={"hindsight": server},')
    lines.append('        allowed_tools=["mcp__hindsight__*"],')
    lines.append('        model="sonnet",')
    lines.append('        permission_mode="bypassPermissions",')
    lines.append("    )")

    if system:
        lines.append(f"    options.system_prompt = {system!r}")
    elif use_hooks:
        lines.append('    options.system_prompt = "You are a helpful coding assistant."')

    if use_hooks:
        lines.append("    options.hooks = hooks")

    lines.append("    result_text = None")
    lines.append(f"    async for msg in query(prompt={prompt!r}, options=options):")
    lines.append("        if hasattr(msg, 'result'):")
    lines.append("            result_text = msg.result")
    lines.append("    print(result_text or '')")
    lines.append("")
    lines.append("asyncio.run(main())")

    script = "\n".join(lines)

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(script)
        tmp_path = f.name

    try:
        for attempt in range(retries):
            result = subprocess.run(
                [sys.executable, tmp_path],
                capture_output=True, text=True, timeout=300,
                env={**os.environ},
            )
            if result.returncode == 0:
                return result.stdout.strip()
            # Rate limited — wait and retry
            wait = 30 * (attempt + 1)
            print(f"Attempt {attempt + 1}/{retries} failed, retrying in {wait}s...")
            time.sleep(wait)

        print("STDERR:", result.stderr[-1000:])
        return result.stdout.strip()
    finally:
        os.unlink(tmp_path)


# Store some preferences
result = run_agent(
    "Store the following into memory using the retain tool:\n"
    "- I prefer Python with type hints and async/await patterns\n"
    "- My team uses pytest for testing with pytest-asyncio\n"
    "- We follow conventional commits (feat:, fix:, chore:)\n"
    "- Our API framework is FastAPI with Pydantic v2 models"
)
print("Agent result:", result)

## 6. Recall Memories in a New Session

Simulate a fresh session — the agent has no conversation history, but can recall from memory:

In [ ]:
# New session — no prior context
result = run_agent(
    "What testing framework does my team use? "
    "Search your memory first before answering.",
    system="You are a helpful coding assistant. Always check memory before answering questions about the user.",
)
print("Agent result:", result)

## 7. Reflect for Deeper Synthesis

Use reflect when you need reasoned analysis across all stored memories:

In [ ]:
result = run_agent(
    ("Use the reflect tool to synthesize what you know about me, with explicit "
     "attention to: programming languages and patterns, testing frameworks, "
     "API/framework choices, and commit conventions. Do not skip any of these "
     "categories — call out each one you have memories about, and say so if a "
     "category has none."),
)
print("Agent result:", result)

## 8. Run Agent with Automatic Memory Hooks

Hooks inject memory automatically — no explicit tool calls needed. The agent gets relevant memories injected as system context before each prompt, and its results are auto-retained for future sessions:

In [ ]:
# The agent receives past memories automatically via hooks — no tool call needed
result = run_agent(
    "What patterns should I follow when writing pytest tests for my FastAPI endpoints?",
    use_hooks=True,
)
print("Agent result:", result)

The agent received your team's testing preferences via auto-recall before it even started working. And its result was auto-retained for future sessions.

## 9. Run Again to See Knowledge Compound

Each session adds to the knowledge base. Run the agent again with a related prompt:

In [ ]:
result = run_agent(
    ("Recall what you know about my commit conventions, then answer: what "
     "commit message format should I use for this test file I just created?"),
    use_hooks=True,
)
print("Agent result:", result)

The agent recalls the conventional commits preference from earlier — even though it was stored in a completely different session.

## Cleanup

Delete the bank created during this notebook:

In [ ]:
hindsight.delete_bank(bank_id=BANK_ID)
print(f"Deleted bank '{BANK_ID}'.")